# RecursiveGCD (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_cpp.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `RecursiveGCD`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono variabili globali, e la traccia si attiva con `trace = true`.

## Massimo comun divisore: RecursiveGCD

**Input:** due numeri naturali $n \ge 1$ e $m \ge 0$. **Output:** $\gcd(n, m)$. Ricorsione di coda: ogni chiamata ne fa al più una, l'albero è un cammino. Al più $2\lfloor\log_2 m\rfloor + 3$ chiamate, $O(\log_2 m)$ passi.

Pseudocodice: sezione 14.6 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [1]:
%%writefile recursive_gcd.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
long long calls = 0;    // numero di chiamate = nodi dell'albero delle chiamate
int maxDepth = 0;       // profondita' della ricorsione = altezza massima dello stack
long long mults = 0;    // moltiplicazioni (FastPower)
long long moves = 0;    // mosse (Hanoi)
int ev = 0;             // contatore degli eventi della traccia
int stk[1000];          // lo stack delle chiamate: gli argomenti delle chiamate attive (al massimo 1000)
int sp = 0;             // numero di chiamate attive
long long seen[64];     // seen[x] = numero di chiamate con argomento x
bool trace = false;     // true: stampa la traccia dell'esecuzione

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // come sprintf, ma restituisce una stringa
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // la chiamata termina: il suo frame esce dallo stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // riga numerata della traccia, con lo stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

int FloorLog2(int m)                               // floor(log2 m) con un ciclo (solo per il limite sulle chiamate)
{
    int k = 0;
    while (m > 1)
    {
        m = m / 2;
        k = k + 1;
    }
    return k;
}

// RecursiveGCD(n, m)
int RecursiveGCD(int n, int m)
{
    Push(m); cnt[2]++;
    if (m == 0)
    {
        cnt[3]++;
        if (trace) Say(Str("RecursiveGCD(%d, %d)", n, m), Str("caso base, restituisce %d", n));
        Pop();
        return n;
    }
    if (trace) Say(Str("RecursiveGCD(%d, %d)", n, m), Str("%d mod %d = %d: chiama RecursiveGCD(%d, %d)", n, m, n % m, m, n % m));
    cnt[5]++;
    int g = RecursiveGCD(m, n % m);
    if (trace) Say(Str("RecursiveGCD(%d, %d)", n, m), Str("restituisce %d", g));
    Pop();
    return g;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempi delle dispense: RecursiveGCD(48, 18) e RecursiveGCD(252, 105), alberi delle chiamate (cammini)
    int pairs[][2] = {{48, 18}, {252, 105}};
    for (auto& p : pairs)
    {
        Reset();
        trace = true;
        printf("%-28s| %s\n", "chiamata", "azione");
        int g = RecursiveGCD(p[0], p[1]);
        trace = false;
        printf("gcd(%d, %d) = %d, chiamate = %lld, S = %lld\n", p[0], p[1], g, calls, Total());
        printf("\n");
    }

    // Numero di chiamate (ogni chiamata esegue 2 righe, quindi S = 2 chiamate) e limite 2 floor(log2 m) + 3
    printf("      n |       m |  gcd | chiamate |   S | 2 floor(log2 m) + 3\n");
    int more[][2] = {{48, 18}, {252, 105}, {89, 55}, {144, 89}, {1000, 7}, {1000000, 999999}};
    for (auto& p : more)
    {
        Reset();
        int g = RecursiveGCD(p[0], p[1]);
        printf("%7d | %7d | %4d | %8lld | %3lld | %d\n", p[0], p[1], g, calls, Total(), 2 * FloorLog2(p[1]) + 3);
    }
    return 0;
}
#endif

Writing recursive_gcd.cpp


In [2]:
!g++ -std=c++17 -O2 -o recursive_gcd recursive_gcd.cpp && ./recursive_gcd

chiamata                    | azione
RecursiveGCD(48, 18)        | 48 mod 18 = 12: chiama RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: chiama RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: chiama RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | caso base, restituisce 6
    RecursiveGCD(12, 6)     | restituisce 6
  RecursiveGCD(18, 12)      | restituisce 6
RecursiveGCD(48, 18)        | restituisce 6
gcd(48, 18) = 6, chiamate = 4, S = 8

chiamata                    | azione
RecursiveGCD(252, 105)      | 252 mod 105 = 42: chiama RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: chiama RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(42, 21)    | restituisce 21
  RecursiveGCD(105, 42)     | restituisce 21
RecursiveGCD(252, 105)      | restituisce 21
gcd(252, 105) = 21, chiamate = 4, S = 8

      n |       m |  gcd | ch

**Prova tu.** Cambia i due numeri; il caso peggiore sono due numeri di Fibonacci consecutivi.

In [3]:
%%writefile prova_recursive_gcd.cpp
#define SOLO_LIBRERIA   // non eseguire il main di recursive_gcd.cpp
#include "recursive_gcd.cpp"

int main()
{
    int n = 1071, m = 462;        // <-- cambia i due numeri
    Reset();
    trace = true;
    printf("%-28s| %s\n", "chiamata", "azione");
    int g = RecursiveGCD(n, m);
    trace = false;
    printf("gcd(%d, %d) = %d, chiamate = %lld\n", n, m, g, calls);
    return 0;
}

Writing prova_recursive_gcd.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_recursive_gcd prova_recursive_gcd.cpp && ./prova_recursive_gcd

chiamata                    | azione
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: chiama RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: chiama RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(147, 21)   | restituisce 21
  RecursiveGCD(462, 147)    | restituisce 21
RecursiveGCD(1071, 462)     | restituisce 21
gcd(1071, 462) = 21, chiamate = 4


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).